In [1]:
import re
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

In [2]:
BASE_DIR = Path(r"D:\Final_GRAG")
REPORT_UNITS_DIR = BASE_DIR / "metadata" / "report_units"

csv_files = sorted(REPORT_UNITS_DIR.glob("*/gri_content_index.csv"))
print(len(csv_files))

28


## Phân tích trường Location

### Constants

In [3]:
MONTHS_ABBR = {
    'jan', 'feb', 'mar', 'apr', 'may', 'jun',
    'jul', 'aug', 'sep', 'oct', 'nov', 'dec',
}

# Cụm từ ngắn KHÔNG phải số trang (Not disclosed, N/A, dấu gạch ngang, ...)
NON_PAGE_RE = re.compile(
    r'^('
    r'not\s+disclosed|not\s+applicable|not\s+available|not\s+reported|'
    r'n/?a|none(?:\s+material)?|yes|'
    r'no\s+restatements?|no\s+incidents?|no\s+significant\b|no\s+known\b|'
    r'no\s+financial\b|no\s+negati\w*|'
    r'-{1,3}|\u2014'
    r')$',
    re.IGNORECASE,
)

# Mẫu cho một token số: số đơn hoặc range ("12", "12-15", "12 to 15")
_N = r'\d+'
_RANGE = rf'{_N}(?:\s*[-\u2013]\s*{_N}|\s+to\s+{_N})?'
# Biểu thức tổng quát: nhiều token cách nhau bằng dấu phẩy
NUM_EXPR = rf'{_RANGE}(?:\s*,\s*{_RANGE})*'

### Trích xuất số trang

In [4]:
def parse_number_text(text):
    """
    Tách chuỗi số / range thành list các số trang.
    Ví dụ: '9-16', '87, 89,90', '43 to 44'.
    """
    pages = []
    # Coi '&' / 'and' giữa các số như dấu phẩy
    text = re.sub(r'\s+(?:and|&)\s+', ',', text, flags=re.IGNORECASE)
    
    for token in re.split(r'[,;]', text):
        token = token.strip()
        if not token:
            continue
        
        # Range với 'to':  "35 to 37"
        m = re.match(r'^(\d+)\s+to\s+(\d+)$', token, re.IGNORECASE)
        if m:
            s, e = int(m.group(1)), int(m.group(2))
            if 0 < s <= e < 10000:
                pages.extend(range(s, e + 1))
            continue
        
        # Range với hyphen / en-dash:  "9-16", "120 - 121"
        m = re.match(r'^(\d+)\s*[-\u2013]\s*(\d+)$', token)
        if m:
            s, e = int(m.group(1)), int(m.group(2))
            if 0 < s <= e < 10000:
                pages.extend(range(s, e + 1))
            continue
        
        # Số đơn
        m = re.match(r'^(\d+)$', token)
        if m:
            pages.append(int(m.group(1)))
    
    return pages


def extract_pages_from_string(s):
    """
    Trích các số trang từ chuỗi Location.
    
    Chiến lược:
      1) Tìm các marker (p., pp., Page, pages, pg) rồi parse số đứng sau.
      2) Nếu không có marker, parse cả chuỗi như danh sách số thuần.
    """
    pages = []
    
    # "p. 12", "pp.9-16", "pp. 42, 44"
    for m in re.finditer(rf'\bpp?\s*\.?\s*({NUM_EXPR})', s, re.IGNORECASE):
        pages.extend(parse_number_text(m.group(1)))
    
    # "Page 12", "pages 18 - 34"
    for m in re.finditer(rf'\bpages?\s+({NUM_EXPR})', s, re.IGNORECASE):
        pages.extend(parse_number_text(m.group(1)))
    
    # "pg38", "pg 49"  (Bapco-style compact prefix)
    for tok in re.findall(rf'\bpg\s*({_N})', s, re.IGNORECASE):
        pages.extend(parse_number_text(tok))
    
    if pages:
        return sorted(set(pages))
    
    # Chuỗi số thuần: chỉ digit, dấu phẩy, khoảng trắng, hyphen
    if re.match(r'^[\d\s,;\u2013\-]+$', s.strip()):
        pages.extend(parse_number_text(s))
    
    return sorted(set(pages))

### Phân loại Location

In [5]:
def classify_pages(pages):
    """Quy ra location_type từ list số trang."""
    if not pages:
        return 'external_document_reference', [], 0
    if len(pages) == 1:
        return 'page_single', pages, 1
    # Chuỗi liên tiếp → page_range
    if pages[-1] - pages[0] + 1 == len(pages):
        return 'page_range', pages, 1
    return 'page_list', pages, 1


def parse_location(loc, report_name=''):
    """Trả về (location_type, page_list, is_location_supported) cho một giá trị Location."""
    # Empty / NaN
    if pd.isna(loc) or str(loc).strip() == '':
        return 'external_document_reference', [], 0
    
    s = str(loc).strip()
    
    # AR cross-reference dạng "AR 27" (Bangchak Annual Report)
    if re.match(r'^AR\s+\d+$', s) and 'Bangchak' in report_name:
        return 'external_document_reference', [], 0
    
    # Artefact Excel: "Aug-16", "Nov-18" — số trang bị Excel hiểu nhầm thành ngày
    parts = s.split('-', 1)
    if len(parts) == 2 and parts[0].strip().lower() in MONTHS_ABBR and parts[1].strip().isdigit():
        return 'external_document_reference', [], 0
    
    # Cụm từ ngắn không phải số trang ("Not disclosed", "N/A", ...)
    if NON_PAGE_RE.match(s):
        return 'external_document_reference', [], 0
    
    # Cross-reference dạng "Please refer to ... Annual Report"
    if re.search(r'\bannual\s+report\b', s, re.IGNORECASE):
        return 'external_document_reference', [], 0
    
    pages = extract_pages_from_string(s)
    return classify_pages(pages)

## Xử lý CSV → JSON

### Tiện ích

In [ ]:
def extract_disclosure_id(text):
    """Trích mã GRI dạng '2-1', '201-1', '3-3' từ cột Disclosure."""
    if pd.isna(text) or str(text).strip() == '':
        return None
    m = re.search(r'\b(\d+-\d+)\b', str(text))
    return m.group(1) if m else None


def clean_str(val):
    """Strip; trả về None cho giá trị rỗng / NaN."""
    if pd.isna(val):
        return None
    s = str(val).strip()
    return s if s else None

### Build records từ một report

In [7]:
REQUIRED_COLS = [
    'Material Topic',
    'GRI Standard / Other Source',
    'Disclosure',
    'Location',
    'Requirement(s) Omitted',
    'Omission - Reason',
    'Omission - Explanation',
    'GRI Sector Standard Ref. No.',
]


def normalize_columns(df):
    """Đồng nhất tên cột và đảm bảo các cột bắt buộc đều tồn tại."""
    df = df.copy()
    df.columns = df.columns.str.strip()
    
    # Một số report dùng tên cột rút gọn
    df = df.rename(columns={
        'Ref. No.': 'GRI Sector Standard Ref. No.',
    })
    
    # Layout cũ tách Section / Page → coi cột Page là Location
    if 'Section / Report Reference' in df.columns and 'Page' in df.columns:
        df = df.rename(columns={'Page': 'Location'})
        df = df.drop(columns=['Section / Report Reference'])
    
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = ''
    
    return df


def build_records(df, report_name):
    """Chuyển DataFrame Content Index thành list dict theo schema target."""
    df = normalize_columns(df)
    # Material Topic chỉ ghi ở dòng đầu của nhóm → forward-fill
    mt_filled = df['Material Topic'].replace('', pd.NA).ffill()
    
    records = []
    for i in range(len(df)):
        row = df.iloc[i]
        loc_raw = clean_str(row['Location'])
        loc_type, page_list, is_supported = parse_location(loc_raw or '', report_name)
        
        records.append({
            "report_name": report_name,
            "material_topic": clean_str(mt_filled.iloc[i]),
            "gri_standard": clean_str(row['GRI Standard / Other Source']),
            "disclosure_id": extract_disclosure_id(row['Disclosure']),
            "location_raw": loc_raw,
            "location_type": loc_type,
            "page_list": page_list,
            "is_location_supported": is_supported,
            "omission_requirement_omitted": clean_str(row['Requirement(s) Omitted']),
            "omission_reason": clean_str(row['Omission - Reason']),
            "omission_explanation": clean_str(row['Omission - Explanation']),
            "gri_sector_standard_ref_no": clean_str(row['GRI Sector Standard Ref. No.']),
            "type": "GRI Content index table",
        })
    
    return records

## Chạy pipeline

In [8]:
summary = []

for csv_path in tqdm(csv_files, desc="Processing reports"):
    report_name = csv_path.parent.name
    
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False).reset_index(drop=True)
    records = build_records(df, report_name)
    
    # Lưu JSON cạnh CSV gốc
    output_path = csv_path.with_suffix('.json')
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    
    # Thống kê location_type cho từng report
    type_counts = {}
    for r in records:
        t = r['location_type']
        type_counts[t] = type_counts.get(t, 0) + 1
    
    summary.append({
        'report': report_name,
        'records': len(records),
        **type_counts,
    })

print(f"Done. {sum(s['records'] for s in summary)} records across {len(summary)} reports.")


Processing reports:   0%|          | 0/28 [00:00<?, ?it/s]


Processing reports:  54%|█████▎    | 15/28 [00:00<00:00, 148.84it/s]


Processing reports: 100%|██████████| 28/28 [00:00<00:00, 163.51it/s]

Done. 3197 records across 28 reports.


## Kiểm tra kết quả

In [ ]:
summary_df = pd.DataFrame(summary).fillna(0)
count_cols = ['records', 'page_single', 'page_range', 'page_list', 'external_document_reference']
for c in count_cols:
    if c in summary_df.columns:
        summary_df[c] = summary_df[c].astype(int)

summary_df

=== Phân bố location_type theo report ===


,report,records,page_single,page_range,external_document_reference,page_list
0,Bangchak2024,141,44,79,18,0
1,Bangchak2025,129,44,67,18,0
2,Bapco2023,101,81,0,11,9
3,Bapco2024,104,81,1,11,11
4,Bunduq2023,119,56,18,34,11
5,Desfa2023,86,37,12,8,29
6,Desfa2024,83,25,20,13,25
7,Energean2023,120,1,0,119,0
8,Energean2024,115,48,19,17,31
9,HPCL2023,147,87,47,3,10
